# 08 · Monitoring

**Role of this layer.** Every stage of every run (and every replay) appends one
JSON object to `runs/events.jsonl`. The log is append-only, so it doubles as an
audit trail and could be shipped to a log platform as is.

The Monitoring page reads two sources, and the functions below (the ones behind
the page) turn them into numbers:
- the **run records**, one per run in its final state: activity, human review,
  tokens and cost, latency and failures;
- the **event log**, for what can happen several times in a run: guardrail
  firings, capability executions and replays.

To make sure there is something to show, this notebook first runs the five example
workflows in mock mode (approving only 1 and 2, as `cli demo` does).

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.engine import run_request
from quantlab.mock_scenarios import SCENARIOS

for scenario in SCENARIOS:
    session = run_request(
        scenario.request, list(scenario.dataset_ids), "mock", approve=scenario.auto_approve
    )
    print(scenario.id, session.status)

chf_safe_haven_success completed
spread_frequency_mismatch_block blocked
arima_forecast_block blocked
prompt_injection_unknown_capability blocked


buy_the_dip_tech completed


In [3]:
from quantlab.lineage import load_runs
from quantlab.monitoring import activity, monitored_runs, read_events

records = load_runs(settings.runs_dir)
events = read_events(settings.events_path)
counted = monitored_runs(records)
print(len(records), "run records,", len(counted), "counted by the page,", len(events), "events")
pd.Series(activity(records))

217 run records, 41 counted by the page, 2614 events


total                      217.000000
completed                   87.000000
blocked                     38.000000
planner_refused             46.000000
failed                       0.000000
rejected                     3.000000
completed_share              0.400922
blocked_share                0.175115
planner_refused_share        0.211982
failed_share                 0.000000
rejected_share               0.013825
completed_percent           40.000000
blocked_percent             18.000000
planner_refused_percent     21.000000
failed_percent               0.000000
rejected_percent             1.000000
dtype: float64

The page counts only the decided runs started on the Analysis page
(`monitored_runs`, printed above), and replays from every source: these headless runs
are not among them. To show what each function computes, the cells below apply them
to every run record.

`activity` splits the runs into five outcomes that do not overlap (completed, blocked
by a deterministic guardrail, refused by the planner with P01, failed, rejected by the
reviewer), so they add up to the total; `_percent` holds the whole percentages the
page shows, which add up to 100.

In [4]:
from quantlab.monitoring import (
    completed_with_warnings,
    grounded_narratives,
    replay_results,
    review_outcomes,
)

pd.Series(
    {
        **review_outcomes(records),
        **replay_results(events),
        **grounded_narratives(records),
        **completed_with_warnings(records),
    }
)

approved               84.000000
rejected                3.000000
approval_rate           0.965517
edited                  7.000000
edit_rate               0.083333
approval_median_s       0.033000
approval_p95_s         70.536750
replays                11.000000
matching               11.000000
replay_match_rate       1.000000
narratives             87.000000
grounded               87.000000
grounded_rate           1.000000
completed              87.000000
with_warnings          27.000000
with_warnings_share     0.310345
dtype: float64

Every guardrail and every capability, zeros included, most frequent first:

In [5]:
from quantlab.monitoring import capability_counts, guardrail_counts

pd.DataFrame(guardrail_counts(events))

,code,name,action,count
0,V10,MATERIAL_ASSUMPTION,APPROVE,429
1,V09,LOOK_AHEAD,APPROVE,57
2,P01,PLANNER_REFUSED,BLOCK,46
3,E04,ALIGNMENT_LOSS,WARN,42
4,V03,INVALID_PARAM,BLOCK,30
5,V08,FREQUENCY_MISMATCH,BLOCK,29
6,V02,UNKNOWN_CAPABILITY,BLOCK,18
7,R01,REQUEST_TOO_LONG,BLOCK,1
8,V01,INVALID_PLAN,BLOCK,0
9,V04,TYPE_MISMATCH,BLOCK,0


In [6]:
pd.DataFrame(capability_counts(events))

,capability,code,count
0,load_series,C01,116
1,resample,C02,98
2,flag_regime,C07,83
3,forward_return,C08,63
4,hac_mean_difference,C10,63
5,compare_groups,C09,54
6,compute_returns,C03,46
7,rolling_volatility,C06,25
8,spread,C05,10
9,difference,C04,0


Cost per run, from the tokens the API reported and the prices in
`config.LLM_PRICES_PER_MILLION_TOKENS`. Mock runs call no LLM and are left out, so
in mock mode the table has no runs and the cache saved nothing:

In [7]:
from quantlab.config import LLM_PRICES_PER_MILLION_TOKENS
from quantlab.monitoring import cache_savings, cost_table

display(pd.DataFrame(cost_table(records, LLM_PRICES_PER_MILLION_TOKENS)))
pd.Series(cache_savings(records, LLM_PRICES_PER_MILLION_TOKENS))

,stage,runs,input_mean,input_p95,output_mean,output_p95,cost_mean,cost_p95,cost_total
0,planner,45,3714.022222,5264.4,377.733333,908.4,0.000560,0.000906,0.025212
1,narrator,45,495.066667,1655.6,62.911111,187.0,0.000081,0.000253,0.003643
2,total,45,4209.088889,6156.6,440.644444,1048.8,0.000641,0.001121,0.028855


planner_calls             45.000000
cache_hits                 9.000000
hit_rate                   0.200000
saved_input_tokens     42023.000000
saved_output_tokens     5517.000000
saved_cost                 0.006961
dtype: float64

Latency per stage (mean and 95th percentile, milliseconds), and the most recent
failures with the stage and the guardrail that stopped them:

In [8]:
from quantlab.monitoring import latency_table, recent_failures

pd.DataFrame(latency_table(records))

,stage,runs,mean_ms,p95_ms,recorded
0,Request check,0,NaN,NaN,False
1,Planner,216,1010.565278,7896.600,True
2,Validator,216,0.143889,0.360,True
3,Executor,87,122.778391,191.869,True
4,Narrator,87,603.067816,3072.820,True


In [9]:
pd.DataFrame(recent_failures(records, limit=5))

,time,run_id,status,stage,code,message
0,2026-10-19T10:16:12.904Z,20261019-101612-9ba0c7,blocked,Validator,V02,The plan has blocking issues: V02:s2
1,2026-10-19T10:16:12.877Z,20261019-101612-876cd5,blocked,Planner,P01,The plan has blocking issues: P01:plan
2,2026-10-19T10:16:12.847Z,20261019-101612-b7ec83,blocked,Validator,V08,The plan has blocking issues: V08:s4
3,2026-10-19T09:52:13.942Z,20261019-095213-6fd8c3,blocked,Validator,V02,The plan has blocking issues: V02:s2
4,2026-10-19T09:52:13.914Z,20261019-095213-8b2a65,blocked,Planner,P01,The plan has blocking issues: P01:plan
